
# Regression Models with Keras


## Introduction


As we've studied, despite the popularity of more powerful libraries such as PyToch and TensorFlow, they are not easy to use and have a steep learning curve. So, for people who are just starting to learn deep learning, there is no better library to use other than the Keras library. 

Keras is a high-level API for building deep learning models. It has gained favor for its ease of use and syntactic simplicity facilitating fast development. As you will see in this exercise and the other exercises in this course, building a very complex deep learning network can be achieved with Keras with only few lines of code. You will appreciate Keras even more, once you learn how to build deep models using PyTorch and TensorFlow in the other courses.

So, in this exercise, you will learn how to use the Keras library to build a regression model.


## Objectives for this Notebook    
* How to use the Keras library to build a regression model
* Download and clean the data set
* Build a neural network
* Train and test the network     



<h2>Table of Contents</h2>


<div class="alert alert-block alert-info" style="margin-top: 20px">

<font size = 4>
1. <a href="#Download-and-Clean-the-Data-Set">Download and Clean the Data Set</a><br>
2. <a href="#Import-Keras-Packages">Import Keras Packages</a><br>
3. <a href="#Build-a-Neural-Network">Build a Neural Network</a><br>
4. <a href="#Train-and-Test-the-Network">Train and Test the Network</a><br>  

</font>
</div>


Let's start by importing the <em>pandas</em> and the Numpy libraries.


#### To use Keras, you will also need to install a backend framework – such as TensorFlow.

If you install TensorFlow 2.16 or above, it will install Keras by default.

We are using the CPU version of tensorflow since we are dealing with smaller datasets. 
You may install the GPU version of tensorflow on your machine to accelarate the processing of larger datasets


#### Suppress the tensorflow warning messages
We use the following code to  suppress the warning messages due to use of CPU architechture for tensoflow.

You may want to **comment out** these lines if you are using the GPU architechture


In [1]:
import os
## os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

In [2]:
import pandas as pd
import numpy as np
import keras

import warnings
warnings.simplefilter('ignore', FutureWarning)

We will be playing around with the same dataset that we used in the videos.

<strong>The dataset is about the compressive strength of different samples of concrete based on the volumes of the different ingredients that were used to make them. Ingredients include:</strong>

* Cement
* Blast furnace slag
* Fly ash
* Water
* Superplasticizer
* Coarse aggregate
* Fine aggregate


## Download and Clean the Data Set


Let's download the data and read it into a <em>pandas</em> dataframe.


In [4]:
concrete_data = pd.read_csv("4.1_concrete_data.csv")
concrete_data.head()

,Cement,Blast Furnace Slag,Fly Ash,Water,Superplasticizer,Coarse Aggregate,Fine Aggregate,Age,Strength
0,540.0,0.0,0.0,162.0,2.5,1040.0,676.0,28,79.99
1,540.0,0.0,0.0,162.0,2.5,1055.0,676.0,28,61.89
2,332.5,142.5,0.0,228.0,0.0,932.0,594.0,270,40.27
3,332.5,142.5,0.0,228.0,0.0,932.0,594.0,365,41.05
4,198.6,132.4,0.0,192.0,0.0,978.4,825.5,360,44.30


So the first concrete sample has 540 cubic meter of cement, 0 cubic meter of blast furnace slag, 0 cubic meter of fly ash, 162 cubic meter of water, 2.5 cubic meter of superplaticizer, 1040 cubic meter of coarse aggregate, 676 cubic meter of fine aggregate. Such a concrete mix which is 28 days old, has a compressive strength of 79.99 MPa. 


#### Let's check how many data points we have


In [5]:
concrete_data.shape

(1030, 9)

So, there are approximately 1000 samples to train our model on. Because of the few samples, we have to be careful not to overfit the training data.


Let's check the dataset for any missing values.


In [6]:
concrete_data.describe()

,Cement,Blast Furnace Slag,Fly Ash,Water,Superplasticizer,Coarse Aggregate,Fine Aggregate,Age,Strength
count,1030.000000,1030.000000,1030.000000,1030.000000,1030.000000,1030.000000,1030.000000,1030.000000,1030.000000
mean,281.167864,73.895825,54.188350,181.567282,6.204660,972.918932,773.580485,45.662136,35.817961
std,104.506364,86.279342,63.997004,21.354219,5.973841,77.753954,80.175980,63.169912,16.705742
min,102.000000,0.000000,0.000000,121.800000,0.000000,801.000000,594.000000,1.000000,2.330000
25%,192.375000,0.000000,0.000000,164.900000,0.000000,932.000000,730.950000,7.000000,23.710000
50%,272.900000,22.000000,0.000000,185.000000,6.400000,968.000000,779.500000,28.000000,34.445000
75%,350.000000,142.950000,118.300000,192.000000,10.200000,1029.400000,824.000000,56.000000,46.135000
max,540.000000,359.400000,200.100000,247.000000,32.200000,1145.000000,992.600000,365.000000,82.600000


In [7]:
concrete_data.isnull().sum()

Cement                0
Blast Furnace Slag    0
Fly Ash               0
Water                 0
Superplasticizer      0
Coarse Aggregate      0
Fine Aggregate        0
Age                   0
Strength              0
dtype: int64

The data looks very clean and is ready to be used to build our model.


#### Split data into predictors and target


The target variable in this problem is the concrete sample strength. Therefore, our predictors will be all the other columns.


In [8]:
concrete_data_columns = concrete_data.columns

In [9]:
predictors = concrete_data[concrete_data_columns[concrete_data_columns != 'Strength']] # all columns except Strength
target = concrete_data['Strength'] # Strength column

<a id="item2"></a>


Let's do a quick sanity check of the predictors and the target dataframes.


In [10]:
predictors.head()

,Cement,Blast Furnace Slag,Fly Ash,Water,Superplasticizer,Coarse Aggregate,Fine Aggregate,Age
0,540.0,0.0,0.0,162.0,2.5,1040.0,676.0,28
1,540.0,0.0,0.0,162.0,2.5,1055.0,676.0,28
2,332.5,142.5,0.0,228.0,0.0,932.0,594.0,270
3,332.5,142.5,0.0,228.0,0.0,932.0,594.0,365
4,198.6,132.4,0.0,192.0,0.0,978.4,825.5,360


In [11]:
target.head()

0    79.99
1    61.89
2    40.27
3    41.05
4    44.30
Name: Strength, dtype: float64

Finally, the last step is to normalize the data by substracting the mean and dividing by the standard deviation.


In [12]:
predictors_norm = (predictors - predictors.mean()) / predictors.std()
predictors_norm.head()

,Cement,Blast Furnace Slag,Fly Ash,Water,Superplasticizer,Coarse Aggregate,Fine Aggregate,Age
0,2.476712,-0.856472,-0.846733,-0.916319,-0.620147,0.862735,-1.217079,-0.279597
1,2.476712,-0.856472,-0.846733,-0.916319,-0.620147,1.055651,-1.217079,-0.279597
2,0.491187,0.795140,-0.846733,2.174405,-1.038638,-0.526262,-2.239829,3.551340
3,0.491187,0.795140,-0.846733,2.174405,-1.038638,-0.526262,-2.239829,5.055221
4,-0.790075,0.678079,-0.846733,0.488555,-1.038638,0.070492,0.647569,4.976069


Let's save the number of predictors to *n_cols* since we will need this number when building our network.


In [13]:
n_cols = predictors_norm.shape[1] # number of predictors

<a id="item1"></a>


##  Import Keras Packages

##### Let's import the rest of the packages from the Keras library that we will need to build our regression model.


In [14]:
from keras.models import Sequential
from keras.layers import Dense
from keras.layers import Input

## Build a Neural Network


Let's define a function that defines our regression model for us so that we can conveniently call it to create our model.


In [15]:
# define regression model
def regression_model():
    # create model
    model = Sequential()
    model.add(Input(shape=(n_cols,)))
    model.add(Dense(50, activation='relu'))
    model.add(Dense(50, activation='relu'))
    model.add(Dense(1))
    
    # compile model
    model.compile(optimizer='adam', loss='mean_squared_error')
    return model

The above function create a model that has two hidden layers, each of 50 hidden units.


## Train and Test the Network


Let's call the function now to create our model.


In [16]:
# build the model
model = regression_model()

Next, we will train and test the model at the same time using the *fit* method. We will leave out 30% of the data for validation and we will train the model for 100 epochs.


In [17]:
# fit the model
model.fit(predictors_norm, target, validation_split=0.3, epochs=100, verbose=2)

Epoch 1/100
23/23 - 1s - 38ms/step - loss: 1690.2825 - val_loss: 1167.4550
Epoch 2/100
23/23 - 0s - 5ms/step - loss: 1571.4171 - val_loss: 1056.7606
Epoch 3/100
23/23 - 0s - 6ms/step - loss: 1396.4410 - val_loss: 895.1083
Epoch 4/100
23/23 - 0s - 7ms/step - loss: 1129.2157 - val_loss: 672.8956
Epoch 5/100
23/23 - 0s - 6ms/step - loss: 786.7845 - val_loss: 431.0847
Epoch 6/100
23/23 - 0s - 6ms/step - loss: 463.8715 - val_loss: 249.5610
Epoch 7/100
23/23 - 0s - 5ms/step - loss: 290.7965 - val_loss: 174.5090
Epoch 8/100
23/23 - 0s - 5ms/step - loss: 236.6886 - val_loss: 158.1618
Epoch 9/100
23/23 - 0s - 6ms/step - loss: 219.0580 - val_loss: 157.2203
Epoch 10/100
23/23 - 0s - 4ms/step - loss: 205.6679 - val_loss: 154.7196
Epoch 11/100
23/23 - 0s - 4ms/step - loss: 195.8127 - val_loss: 152.0379
Epoch 12/100
23/23 - 0s - 5ms/step - loss: 188.5728 - val_loss: 151.1664
Epoch 13/100
23/23 - 0s - 4ms/step - loss: 182.9736 - val_loss: 148.6494
Epoch 14/100
23/23 - 0s - 5ms/step - loss: 177.7712 -

<strong>You can refer to this [link](https://keras.io/models/sequential/) to learn about other functions that you can use for prediction or evaluation.</strong>


Feel free to vary the following and note what impact each change has on the model's performance:

1. Increase or decreate number of neurons in hidden layers
2. Add more hidden layers
3. Increase number of epochs


<h3>Practice Exercise 1</h3>


Now using the same dateset,try to recreate regression model featuring five hidden layers, each with 50 nodes and ReLU activation functions, a single output layer, optimized using the Adam optimizer.


In [18]:
def regression_model():
    input_colm = predictors_norm.shape[1] # Number of input features
    # create model
    model = Sequential()
    model.add(Input(shape=(input_colm,)))  # Set the number of input features 
    model.add(Dense(50, activation='relu'))  
    model.add(Dense(50, activation='relu'))
    model.add(Dense(50, activation='relu')) 
    model.add(Dense(50, activation='relu'))
    model.add(Dense(50, activation='relu'))  
    model.add(Dense(1))  # Output layer
    
    # compile model
    model.compile(optimizer='adam', loss='mean_squared_error')
    return model

In [19]:
# build the model
model = regression_model()

In [20]:
# fit the model
model.fit(predictors_norm, target, validation_split=0.3, epochs=100, verbose=2)

Epoch 1/100
23/23 - 1s - 55ms/step - loss: 1630.6198 - val_loss: 1044.0852
Epoch 2/100
23/23 - 0s - 6ms/step - loss: 975.0953 - val_loss: 198.9477
Epoch 3/100
23/23 - 0s - 5ms/step - loss: 295.1621 - val_loss: 196.3866
Epoch 4/100
23/23 - 0s - 5ms/step - loss: 224.6670 - val_loss: 212.8683
Epoch 5/100
23/23 - 0s - 5ms/step - loss: 206.0564 - val_loss: 182.9387
Epoch 6/100
23/23 - 0s - 5ms/step - loss: 185.5961 - val_loss: 182.1656
Epoch 7/100
23/23 - 0s - 5ms/step - loss: 174.5749 - val_loss: 190.7250
Epoch 8/100
23/23 - 0s - 5ms/step - loss: 158.0042 - val_loss: 187.0881
Epoch 9/100
23/23 - 0s - 5ms/step - loss: 141.7395 - val_loss: 186.2469
Epoch 10/100
23/23 - 0s - 5ms/step - loss: 128.2609 - val_loss: 213.2476
Epoch 11/100
23/23 - 0s - 5ms/step - loss: 117.5313 - val_loss: 184.9196
Epoch 12/100
23/23 - 0s - 7ms/step - loss: 98.5557 - val_loss: 157.2233
Epoch 13/100
23/23 - 0s - 6ms/step - loss: 88.8457 - val_loss: 168.8580
Epoch 14/100
23/23 - 0s - 6ms/step - loss: 75.9065 - val_lo

<h3>Practice Exercise 2</h3>


 Train and evaluate the model simultaneously using the fit() method by reserving 10% of the data for validation and training the model for 100 epochs


In [21]:
# build the model
model = regression_model()
model.fit(predictors_norm, target, validation_split=0.1, epochs=100, verbose=2)

Epoch 1/100
29/29 - 1s - 45ms/step - loss: 1522.4998 - val_loss: 965.8321
Epoch 2/100
29/29 - 0s - 6ms/step - loss: 679.7455 - val_loss: 351.9090
Epoch 3/100
29/29 - 0s - 5ms/step - loss: 254.7061 - val_loss: 228.2099
Epoch 4/100
29/29 - 0s - 5ms/step - loss: 204.3571 - val_loss: 206.4710
Epoch 5/100
29/29 - 0s - 5ms/step - loss: 181.5267 - val_loss: 201.3119
Epoch 6/100
29/29 - 0s - 4ms/step - loss: 167.4855 - val_loss: 175.1256
Epoch 7/100
29/29 - 0s - 4ms/step - loss: 153.6631 - val_loss: 169.5811
Epoch 8/100
29/29 - 0s - 4ms/step - loss: 142.4802 - val_loss: 150.5948
Epoch 9/100
29/29 - 0s - 4ms/step - loss: 128.3510 - val_loss: 123.2548
Epoch 10/100
29/29 - 0s - 4ms/step - loss: 110.3681 - val_loss: 104.1253
Epoch 11/100
29/29 - 0s - 5ms/step - loss: 98.1504 - val_loss: 96.0582
Epoch 12/100
29/29 - 0s - 4ms/step - loss: 84.1362 - val_loss: 88.6385
Epoch 13/100
29/29 - 0s - 4ms/step - loss: 74.5237 - val_loss: 63.5871
Epoch 14/100
29/29 - 0s - 4ms/step - loss: 66.4388 - val_loss: 7

Based on the results, we notice that:

- Adding more hidden layers to the model increases its capacity to learn and represent complex relationships within the data. This allows the model to better identify, as a result, the model becomes more effective at fitting the training data and potentially improving its predictions.
- By reducing the proportion of data set aside for validation and using a larger portion for training, the model has access to more examples to learn from. This additional training data helps the model improve its understanding of the underlying trends, which can lead to better overall performance.  


Based on the results, we notice that:

- Adding more hidden layers to the model increases its capacity to learn and represent complex relationships within the data. This allows the model to better identify, as a result, the model becomes more effective at fitting the training data and potentially improving its predictions.
- By reducing the proportion of data set aside for validation and using a larger portion for training, the model has access to more examples to learn from. This additional training data helps the model improve its understanding of the underlying trends, which can lead to better overall performance.  


### Thank you for completing this lab!

This notebook was created by [Alex Aklson](https://www.linkedin.com/in/aklson/). I hope you found this lab interesting and educational. Feel free to contact me if you have any questions!


<!--
## Change Log

|  Date (YYYY-MM-DD) |  Version | Changed By  |  Change Description |
|---|---|---|---|
| 2024-11-20  | 3.0  | Aman  |  Updated the library versions to current |
| 2020-09-21  | 2.0  | Srishti  |  Migrated Lab to Markdown and added to course repo in GitLab |



<hr>

## <h3 align="center"> © IBM Corporation. All rights reserved. <h3/>


## <h3 align="center"> &#169; IBM Corporation. All rights reserved. <h3/>

